In [29]:
import os
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Tạo hàm để đọc file parquet (đọc các file parquet - data sau khi được processed)

In [30]:
def read_parquet_user(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    user_chunk_files = [file for file in files if 'user_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    user_chunk_df = pl.concat([pl.read_parquet(file) for file in user_chunk_files]) if user_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return user_chunk_df

In [31]:
def read_parquet_item(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    item_chunk_files = [file for file in files if 'item_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    item_chunk_df = pl.concat([pl.read_parquet(file) for file in item_chunk_files]) if item_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return item_chunk_df

In [32]:
def read_parquet_transaction(train_path: str):
    # Lấy tất cả các file parquet trong thư mục
    files = [os.path.join(train_path, f) for f in os.listdir(train_path) if f.endswith('.parquet')]
    
    # Phân loại các file theo loại tên
    purchase_chunk_files = [file for file in files if 'purchase_history_daily_chunk' in file]
        
    # Đọc các file riêng biệt thành DataFrame
    purchase_chunk_df = pl.concat([pl.read_parquet(file) for file in purchase_chunk_files]) if purchase_chunk_files else None
        
    # Trả về một dictionary chứa các DataFrame
    return purchase_chunk_df

# Tạo hàm lưu file parquet sau mỗi task

In [33]:
def split_and_save_parquet(df, num_files, output_dir):
    """
    Tách DataFrame thành nhiều file Parquet và lưu vào thư mục đích.
    
    :param df: DataFrame cần tách
    :param num_files: Số lượng file Parquet muốn tách
    :param output_dir: Thư mục lưu các file Parquet
    """
    # Đảm bảo thư mục tồn tại
    os.makedirs(output_dir, exist_ok=True)
    
    # Tính số dòng mỗi file sẽ có
    num_rows = df.height
    rows_per_file = num_rows // num_files

    # Tách DataFrame thành các phần và lưu mỗi phần vào một file Parquet
    for i in range(num_files):
        start_row = i * rows_per_file
        # Đảm bảo phần cuối cùng sẽ chứa tất cả các dòng còn lại
        end_row = (i + 1) * rows_per_file if i < num_files - 1 else num_rows
        
        # Tách phần DataFrame
        split_df = df[start_row:end_row]
        
        # Lưu phần DataFrame vào file .parquet
        file_path = os.path.join(output_dir, f"sale_pers.item_chunk_{i}.parquet")
        split_df.write_parquet(file_path)
        print(f"Đã lưu file: {file_path}")

# Task 2: Xử lý NULL, Xử lý Outlier

## Xử lý null

### Xử lý age_group

In [34]:
df_age = read_parquet_item("./preprocessed-dataset")
df_age.head()

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new,gender_target_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cu…","""Không xác định""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định""","""Bé Gái""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm bằng chất liệu silicone mềm, dẻo và nước đã được chưng cất đảm bảo an…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Miếng Gặm Nướu Papa (CEQ004) (Cá hồng) C…","""Không xác định""","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miếng là sản phẩm dành cho bé 4-8kg đến từ thương hiệu uy tín Merries của…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M 58 miếng là sản phẩm dành cho bé từ 6-11kg đến từ thương hiệu uy tín M…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định"""


#### Ý tưởng 2:

Thống kê

In [35]:
import polars as pl

df = df_age   # hoặc df_filled tùy bạn đang dùng

# Hàm tiện dụng để lấy danh sách unique của 1 cột
def get_unique_list(df, col):
    return (
        df.select(col)
          .unique()
          .sort(col)
          .get_column(col)
          .to_list()
    )

# Lấy tất cả class
age_list    = get_unique_list(df, "age_group_final")

# Gom vào dictionary
category_dict = {
    "age_group": age_list
}

print(len(age_list))

# In ra theo từng dòng, rất dễ đọc
print("\n=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===\n")
for key, value in category_dict.items():
    print(f"{key}: {value}\n")

274

=== DANH SÁCH TẤT CẢ CLASS CỦA TỪNG CATEGORY ===

age_group: ['0-10M', '0-11M', '0-120M', '0-12M', '0-144M', '0-14M', '0-18M', '0-1M', '0-24M', '0-2M', '0-30M', '0-36M', '0-3M', '0-48M', '0-4M', '0-5M', '0-60M', '0-6M', '0-72M', '0-84M', '0-9M', '1-12M', '1-18M', '1-24M', '1-36M', '1-3M', '108-120M', '108-132M', '12-108M', '12-120M', '12-132M', '12-144M', '12-14M', '12-192M', '12-24M', '12-36M', '12-48M', '12-60M', '12-72M', '120-144M', '12M-18M', '12M-4Y', '13-16M', '13-17M', '132-144M', '13M-24M', '14-17M', '18-24M', '18M-24M', '18M-36M', '18M-4Y', '1M-12M', '1M-15M', '1M-3M', '24-120M', '24-36M', '24-48M', '24-60M', '24-72M', '24-96M', '2M-15M', '2M-6M', '3-18M', '3-24M', '3-6M', '36-120M', '36-144M', '36-216M', '36-48M', '36-60M', '36-72M', '3M-12M', '3M-18M', '3M-24M', '3M-6M', '4-24M', '4-30M', '4-6M', '48-60M', '48-72M', '4M-4Y', '4M-6M', '6-12M', '6-144M', '6-15M', '6-18M', '6-24M', '6-30M', '6-36M', '6-60M', '6-72M', '6-84M', '6-8M', '6-9M', '60-72M', '6M-10M', '6M-12M', 

In [36]:
unknown_df = (
    df_age
    .filter(
        (pl.col("age_group_final") == "Không xác định") &
        ~(
            (pl.col("description").is_null() | (pl.col("description") == "Không xác định")) &
            (pl.col("description_new").is_null() | (pl.col("description_new") == "Không xác định"))
        )
    )
    .select([
        "item_id",
        "description",
        "description_new",
        "age_group_final"
    ])
)

unknown_df

item_id,description,description_new,age_group_final
str,str,str,str
"""0020010000094""","""﻿﻿Tã dán Merries size S 82 miếng là sản phẩm dành cho bé 4-8kg đến từ thương hiệu uy tín Merries của…","""Không xác định""","""Không xác định"""
"""0020010000098""","""﻿﻿﻿Bỉm tã quần Merries size M 58 miếng là sản phẩm dành cho bé từ 6-11kg đến từ thương hiệu uy tín M…","""Không xác định""","""Không xác định"""
"""0024181040235""","""Áo thun bé trai tay ngắn CF B078010 Xanh là sản phẩm với chất liệu cotton mềm mại, thấm hút mồ hôi,…","""Không xác định""","""Không xác định"""
"""0020010000150""","""﻿﻿﻿Dòng sản phẩm tã quần Huggies Dry với tinh chất tràm trà tự nhiên sẽ mang đến cho da con cảm giác…","""Không xác định""","""Không xác định"""
"""0020010000151""","""Sản phẩm Tã dán sơ sinh Moony dưới 5kg, 90 miếng với đặc điểm nổi bật: VIỀN ĐIỀU HÒA THÔNG MINH giúp…","""Chi tiết sản phẩm Tên sản phẩm: Bỉm - Tã dán Moony newborn 90 miếng ( Th…","""Không xác định"""
"""0010251040141""","""Chân váy kaki bé gái CF G048017 Xanh của thương hiệu CF, sản xuất tại Việt Nam. Sản phẩm được may bằ…","""Không xác định""","""Không xác định"""
"""0007040040001""","""Được thiết kế thuận tiện cho việc chăm sóc trẻ em. Thiết kế sang trọng, độ bền cao. Bánh xe mềm giúp…","""Chi tiết sản phẩm Tên sản phẩm: Xe ăn bột cao cấp Carrot XP-03, Hồng Trọ…","""Không xác định"""
"""0009051040002""","""﻿Với chất liệu vải 100% cotton, ruột gối làm bằng gòn 100% Polyester tạo cảm giác dể chịu, êm ái, th…","""Chi tiết sản phẩm Tên sản phẩm: Gối Ôm Cho Bé BabyTop Hình Heo Con - 25x60cm …","""Không xác định"""
"""0007080000368""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Tăm Bông Diva Thân Giấy Hộp Tròn - 200 Que …","""Không xác định"""


Hiện tại có 4_719 dòng dữ liệu đang tồn tại `description`, `description_new` có thông tin (bởi vì có các dòng mà cả 2 `description` đều là "Không xác định" hoặc Null)

In [37]:
import textwrap

# Hàm wrap 175 ký tự
def wrap175(text):
    if text is None:
        return ""
    return textwrap.fill(str(text), width=175)

with open("unknown_output.txt", "w", encoding="utf-8") as f:
    for row in unknown_df.select(["item_id", "description", "description_new"]).iter_rows(named=True):
        f.write("-----------------------------------------------------\n")
        f.write(f"item_id: {row['item_id']}\n")

        f.write("description:\n")
        f.write(wrap175(row["description"]) + "\n\n")

        f.write("description_new:\n")
        f.write(wrap175(row["description_new"]) + "\n\n")

print("Đã xuất file: unknown_output.txt")

Đã xuất file: unknown_output.txt


In [38]:
import re

# ============================================================
# 1. CẤU HÌNH CHO CÂN NẶNG (WEIGHT)
#    (Dùng chung POS_CONTEXT, CHILD_WORDS, NEG_STRONG, WINDOW_POS, WINDOW_NEG)
# ============================================================

POS_CONTEXT = [
    "bé", "trẻ", "trẻ em", "em bé",
    "baby", "kid", "child", "children",
    "thiếu nhi", "newborn", "sơ sinh",
    "tháng tuổi", "months old", "years old",
]

CHILD_WORDS = [
    "bé", "trẻ", "trẻ em", "em bé",
    "baby", "kid", "child", "children",
]

NEG_STRONG = [
    # hạn sử dụng / bảo quản
    "sử dụng trong vòng",
    "hạn sử dụng", "thời hạn", "hạn dùng", "hạn sử dụng tốt nhất",
    "bảo quản", "kể từ ngày sản xuất", "sau khi mở nắp",

    # không dùng / không phù hợp cho trẻ
    "không thích hợp cho trẻ",
    "không thích hợp với trẻ",
    "không dùng cho trẻ",
    "không nên dùng cho trẻ",
    "không phù hợp cho trẻ",
    "không phù hợp với trẻ",
    "không sử dụng sản phẩm cho trẻ",
    "không sử dụng cho trẻ",

    # tiếng Anh
    "not suitable for children",
    "do not use for children",
]

# Từ khoá cân nặng nói chung (đã dùng trong code cũ để loại khỏi TUỔI)
WEIGHT_WORDS = [
    "kg", "kilogram", "kilograms", "ký", "kí", "cân", "nặng", "trọng lượng", "weight"
]

# Các cụm liên quan đến khối lượng / trọng lượng tịnh của SẢN PHẨM
# -> dùng làm NEG context cho CÂN NẶNG CỦA BÉ
WEIGHT_PACKAGING_WORDS = [
    "khối lượng tịnh",
    "trọng lượng tịnh",
    "trọng lượng",
    "khối lượng",
    "net weight",
    "gross weight",
    "khối lượng cả bì",
]

# ============================================================
# 3. REGEX PATTERNS CHO CÂN NẶNG
# ============================================================

# 3.1. "sơ sinh ... đến ... 15kg" -> hiểu là 0-15KG
NEWBORN_TO_KG_PATTERN = re.compile(
    r"sơ\s*sinh"
    r".{0,40}?(?:đến|tới|to|-).{0,20}?"
    r"(?P<w>\d{1,2}(?:[.,]\d+)?)\s*(kg|kilogram|kilograms|ký|kí)"
)

# Range: "6-11kg", "9 - 14kg", "từ 3.5kg đến 16kg", "Trẻ từ 10kg-20kg"
WEIGHT_RANGE_PATTERN = re.compile(
    r"(?:từ\s*)?"                                  # optional 'từ'
    r"(?P<w1>\d{1,3}(?:[.,]\d+)?)\s*"              # số thứ nhất
    r"(?:kg|kilogram|kilograms|ký|kí)?\s*"         # optional kg sau số thứ nhất
    r"(?:-|–|—|đến|tới|to)\s*"                     # dấu nối / đến / to
    r"(?P<w2>\d{1,3}(?:[.,]\d+)?)\s*"              # số thứ hai
    r"(kg|kilogram|kilograms|ký|kí)"               # kg cuối
)

# Single với so sánh: "dưới 15kg", "từ 9kg", "trên 12kg", "9kg trở lên"
WEIGHT_SINGLE_PATTERN = re.compile(
    r"(?:(?P<cmp>dưới|under|<|từ|from|trên|hơn|>=|>|over|more than)\s*)?"
    r"(?P<w>\d{1,3}(?:[.,]\d+)?)\s*"
    r"(kg|kilogram|kilograms|ký|kí)"
    r"(?:\s*(?P<suffix>trở lên|\+))?"
)

WINDOW_POS = 20   # window cho POS context
WINDOW_NEG = 80  # window cho NEG context

def has_pos_context(context: str) -> bool:
    return any(tok in context for tok in POS_CONTEXT)

# ============================================================
# 2. HÀM NEG CONTEXT RIÊNG CHO CÂN NẶNG (KHÔNG DÙNG WEIGHT_WORDS NHƯ TUỔI)
# ============================================================

def has_neg_context_weight(context: str) -> bool:
    """
    NEG context cho CÂN NẶNG CỦA BÉ.
    Dùng lại NEG_STRONG, CHILD_WORDS, nhưng KHÔNG loại bỏ mọi chỗ có 'kg' như khi xử lý TUỔI.
    Chỉ coi là NEG nếu liên quan đến TRỌNG LƯỢNG TỊNH / PACK SIZE, hoặc các cảnh báo không dùng cho trẻ.
    """
    ctx = context

    # 1) Các cụm NEG mạnh (hạn sử dụng / bảo quản / ...)
    if any(phrase in ctx for phrase in NEG_STRONG):
        return True

    # 2) "không ... (phù hợp/thích hợp/dùng/sử dụng) ... trẻ"
    if "không" in ctx and any(v in ctx for v in ["phù hợp", "thích hợp", "dùng", "sử dụng"]):
        if any(c in ctx for c in CHILD_WORDS):
            return True

    # 3) "tránh dùng / tránh sử dụng ... cho trẻ"
    if any(start in ctx for start in ["tránh dùng", "tránh sử dụng"]):
        if any(c in ctx for c in CHILD_WORDS):
            return True

    # 4) English NEG liên quan trẻ
    if "not suitable" in ctx and any(c in ctx for c in ["child", "children", "kid", "baby"]):
        return True
    if "do not use" in ctx and any(c in ctx for c in ["child", "children", "kid", "baby"]):
        return True

    # 5) CASE THỜI LƯỢNG "6 tháng đầu" ... (nếu bạn muốn loại những đoạn này)
    if "tháng đầu" in ctx or "tháng đầu đời" in ctx:
        if any(k in ctx for k in ["trong ", "trong vòng", "giai đoạn"]):
            return True

    # 6) PACK SIZE / TRỌNG LƯỢNG TỊNH -> không phải cân nặng của bé
    if any(w in ctx for w in WEIGHT_PACKAGING_WORDS):
        return True

    return False

# ============================================================
# 4. HÀM EXTRACT CÂN NẶNG TỪ MÔ TẢ
# ============================================================

def extract_weight_phrases(desc: str) -> list[str]:
    """
    Trả về list các cụm cân nặng LIÊN QUAN ĐẾN BÉ trong desc, ví dụ:
      - "6-11kg", "9 - 14kg"
      - "từ 3.5kg đến 16kg"
      - "dưới 15kg", "từ 9kg", "9kg trở lên"
      - "Trẻ từ 10kg-20kg"

    Rule:
      - CHỈ nhìn phần TRƯỚC "Lưu ý".
      - Phải có POS context (bé/trẻ/...) trong WINDOW_POS.
      - Nếu trong WINDOW_NEG có NEG context weight (pack size, cấm dùng, ...) thì BỎ.
    """
    if desc is None:
        return []

    full_text = desc.lower()

    # Cắt bỏ phần sau 'lưu ý' (giống hàm extract_age_phrases)
    m_luuy = re.search(r"lưu\s*ý\b", full_text)
    if m_luuy:
        text = full_text[:m_luuy.start()]
    else:
        text = full_text

    matches: list[str] = []
    used_spans: list[tuple[int, int]] = []

    def overlap_span(start: int, end: int) -> bool:
        return any(not (end <= s or start >= e) for (s, e) in used_spans)

    def get_contexts(start_idx: int, end_idx: int) -> tuple[str, str]:
        left_pos = max(0, start_idx - WINDOW_POS)
        right_pos = min(len(text), end_idx + WINDOW_POS)
        context_pos = text[left_pos:right_pos]

        left_neg = max(0, start_idx - WINDOW_NEG)
        right_neg = min(len(text), end_idx + WINDOW_NEG)
        context_neg = text[left_neg:right_neg]

        return context_pos, context_neg

    # 1) RANGE: "từ 3.5kg đến 16kg", "10kg-20kg", "12 - 20kg"
    for m in WEIGHT_RANGE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context_weight(context_neg):
            continue
        matches.append(text[start_idx:end_idx].strip())
        used_spans.append((start_idx, end_idx))

    # 2) SINGLE: "dưới 15kg", "từ 9kg", "9kg trở lên", ...
    for m in WEIGHT_SINGLE_PATTERN.finditer(text):
        start_idx, end_idx = m.span()
        if overlap_span(start_idx, end_idx):
            continue

        context_pos, context_neg = get_contexts(start_idx, end_idx)
        if not has_pos_context(context_pos) or has_neg_context_weight(context_neg):
            continue

        match_text = text[start_idx:end_idx].strip()
        matches.append(match_text)
        used_spans.append((start_idx, end_idx))

    return matches

# ============================================================
# 5. CHUẨN HÓA CÂN NẶNG VỀ DẠNG CANONICAL
# ============================================================

def _to_kg(num_str: str) -> float:
    """
    '9', '9,5', '9.5' -> giá trị kg (float).
    """
    return float(num_str.replace(",", "."))


def _fmt_kg(val: float) -> str:
    """
    Định dạng số kg:
      - 3.0  -> '3'
      - 3.5  -> '3.5'
      - 3.25 -> '3.25'
    """
    x = float(val)
    if x.is_integer():
        return str(int(x))
    s = f"{x:.3f}".rstrip("0").rstrip(".")
    return s



def normalize_weight_phrase(raw: str) -> str | None:
    """
    Chuẩn hóa 1 cụm cân nặng về canonical:

      - '6-11KG'      : 6–11 kg
      - '3.5-16KG'    : 3.5–16 kg
      - '0-15KG'      : dưới 15kg / under 15kg
      - '9KG+'        : từ 9kg / 9kg trở lên / trên 9kg

    Trả về None nếu không chắc chắn.
    """
    if not raw:
        return None

    s = raw.strip().lower()
    s2 = re.sub(r"\s+", " ", s)

    # --------------------------------------------------------
    # 1) RANGE: ưu tiên xử lý TRƯỚC (tránh bị tách thành 2 single)
    #    - 'từ 3.5kg đến 16kg'
    #    - '3,5 đến 6,5kg'
    #    - '10kg-20kg'
    #    - '12 - 20kg'
    # --------------------------------------------------------
    m_range = WEIGHT_RANGE_PATTERN.search(s2)
    if m_range:
        a_str = m_range.group("w1")
        b_str = m_range.group("w2")
        a = _to_kg(a_str)
        b = _to_kg(b_str)
        start = min(a, b)
        end = max(a, b)
        return f"{_fmt_kg(start)}-{_fmt_kg(end)}KG"

    # --------------------------------------------------------
    # 2) UNDER: 'dưới 15kg', 'under 15kg', '< 15kg' -> 0-15KG
    #    (Ở đây 0 chỉ là mốc abstract, không phải cân nặng thực tế)
    # --------------------------------------------------------
    m_under = re.search(
        r"(dưới|under|<)\s*(\d{1,3}(?:[.,]\d+)?)\s*(kg|kilogram|kilograms|ký|kí)",
        s2,
    )
    if m_under:
        w_str = m_under.group(2)
        end_kg = _to_kg(w_str)
        if end_kg <= 0:
            return None
        return f"0-{_fmt_kg(end_kg)}KG"

    # --------------------------------------------------------
    # 3) FROM / OVER / PLUS:
    #    - 'từ 9kg', 'trên 9kg', '9kg trở lên', '9kg+'
    #    -> '9KG+'
    # --------------------------------------------------------
    m_from = re.search(
        r"(từ|from|trên|hơn|>=|>|over|more than)\s*"
        r"(\d{1,3}(?:[.,]\d+)?)\s*"
        r"(kg|kilogram|kilograms|ký|kí)"
        r"(?:\s*(trở lên|\+))?",
        s2,
    )
    if m_from:
        w_str = m_from.group(2)
        start_kg = _to_kg(w_str)
        return f"{_fmt_kg(start_kg)}KG+"

    m_suffix_plus = re.search(
        r"(\d{1,3}(?:[.,]\d+)?)\s*"
        r"(kg|kilogram|kilograms|ký|kí)\s*"
        r"(trở lên|\+)",
        s2,
    )
    if m_suffix_plus:
        w_str = m_suffix_plus.group(1)
        start_kg = _to_kg(w_str)
        return f"{_fmt_kg(start_kg)}KG+"

    # --------------------------------------------------------
    # 4) SINGLE '10kg' đơn lẻ (không 'từ/dưới/trên') -> mơ hồ (pack size?)
    #    -> bỏ qua để giữ độ chắc chắn cao.
    # --------------------------------------------------------
    return None

def _parse_weight_canonical(c: str) -> tuple[float, float | None, bool]:
    """
    Trả (start_kg, end_kg, is_plus)
      - '6KG'      -> (6.0, 6.0, False)
      - '6-11KG'   -> (6.0, 11.0, False)
      - '3.5-16KG' -> (3.5, 16.0, False)
      - '9KG+'     -> (9.0, None, True)
    """
    if c.endswith("KG+"):
        start_str = c[:-3]
        start = float(start_str)
        return start, None, True

    if not c.endswith("KG"):
        raise ValueError(f"Canonical weight phải kết thúc bằng 'KG' hoặc 'KG+': {c}")

    body = c[:-2]
    if "-" in body:
        a_str, b_str = body.split("-", 1)
        start = float(a_str)
        end = float(b_str)
        return start, end, False
    else:
        v = float(body)
        return v, v, False


def normalize_weight_phrase_list(raw_list: list[str] | None) -> list[str]:
    """
    Chuẩn hóa list các cụm cân nặng:
      - Chạy normalize_weight_phrase trên từng phần tử.
      - Loại trùng.
      - Loại các bucket bị bao phủ bởi bucket khác (giống logic tuổi).
    """
    if raw_list is None:
        return []

    # 1) Chuẩn hoá từng cụm
    canon: list[str] = []
    for raw in raw_list:
        norm = normalize_weight_phrase(raw)
        if norm is not None:
            canon.append(norm)

    # 2) unique theo thứ tự
    uniq: list[str] = []
    for c in canon:
        if c not in uniq:
            uniq.append(c)

    if not uniq:
        return []

    # 3) loại các mốc bị "bao phủ"
    parsed = [_parse_weight_canonical(c) for c in uniq]
    keep = [True] * len(uniq)

    for i, (si, ei, plus_i) in enumerate(parsed):
        if not keep[i]:
            continue
        for j, (sj, ej, plus_j) in enumerate(parsed):
            if i == j or not keep[j]:
                continue

            # j bao phủ i?
            if plus_j:
                # j là 'sjKG+' -> [sj, +∞)
                if ei is None:
                    # cả 2 đều '+': giữ cái xuất hiện trước
                    if j < i:
                        keep[i] = False
                        break
                else:
                    if si >= sj:
                        keep[i] = False
                        break
            else:
                # j là khoảng hữu hạn [sj, ej]
                if ei is None:
                    # khoảng vô hạn không bị cover bởi hữu hạn
                    continue
                if sj <= si and ej >= ei:
                    keep[i] = False
                    break

    result = [c for c, k in zip(uniq, keep) if k]
    return result


In [39]:
import textwrap

pl.Config.set_fmt_str_lengths(100)
pl.Config.set_tbl_rows(50)

def wrap175(text):
    if text is None:
        return ""
    return textwrap.fill(str(text), width=175)


df_unknown_kg = (
    df_age
    .filter(pl.col("age_group_final") == "Không xác định")
    .with_columns(
        pl.concat_str(
            [
                pl.col("description").fill_null(""),
                pl.col("description_new").fill_null(""),
            ],
            separator=" "
        )
        .str.to_lowercase()
        .alias("desc_all")
    )
    .with_columns(
        pl.col("desc_all").map_elements(
            extract_weight_phrases,
            return_dtype=pl.List(pl.Utf8)
        ).alias("weight_phrases_raw")
    )
    .with_columns(
        pl.col("weight_phrases_raw").map_elements(
            normalize_weight_phrase_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("weight_ranges_norm")
    )
)

# Lấy tất cả canonical WEIGHT ranges đã detect
weight_norm_list = (
    df_unknown_kg
    .select("weight_ranges_norm")
    .explode("weight_ranges_norm")
    .filter(pl.col("weight_ranges_norm").is_not_null())
    .unique()
    .to_series()
    .to_list()
)

print("Số lượng weight_norm:", len(weight_norm_list))
weight_norm_list

# Xuất ví dụ theo từng WEIGHT_NORM ra file
with open("output_weight_norm.txt", "w", encoding="utf-8") as f:

    for wg in weight_norm_list:
        f.write(f"\n=== VÍ DỤ CHO WEIGHT_NORM = {wg} ===\n\n")
        weight_norm = wg

        examples = (
            df_unknown_kg
            .filter(pl.col("weight_ranges_norm").list.contains(weight_norm))
            .select([
                "item_id",
                "age_group_final",
                "weight_phrases_raw",
                "weight_ranges_norm",
                "description",
                "description_new",
            ])
            .head(20)
        )

        for row in examples.iter_rows(named=True):
            f.write("======================================\n")
            f.write(f"item_id: {row['item_id']}\n")
            f.write(f"age_group_final: {row['age_group_final']}\n")
            f.write(f"weight_phrases_raw: {row['weight_phrases_raw']}\n")
            f.write(f"weight_ranges_norm: {row['weight_ranges_norm']}\n\n")

            f.write("description:\n")
            f.write(wrap175(row["description"]) + "\n\n")

            f.write("description_new:\n")
            f.write(wrap175(row["description_new"]) + "\n\n")

print("Đã xuất file: output_weight_norm.txt")


Số lượng weight_norm: 73
Đã xuất file: output_weight_norm.txt


Tìm sự tương quan của các dòng đã có giá trị và các dòng "không xác định" trong `age_group_final` 

Trước tiên list những dòng vừa có thông tin của `weight` và `age`

In [40]:
import polars as pl

# Bước 1: tạo desc_all + weight_phrases_raw + weight_ranges_norm cho TOÀN bộ df_age
df_age_w = (
    df_age
    .with_columns(
        pl.concat_str(
            [
                pl.col("description").fill_null(""),
                pl.col("description_new").fill_null(""),
            ],
            separator=" "
        )
        .str.to_lowercase()
        .alias("desc_all")
    )
    .with_columns(
        pl.col("desc_all").map_elements(
            extract_weight_phrases,
            return_dtype=pl.List(pl.Utf8)
        ).alias("weight_phrases_raw")
    )
    .with_columns(
        pl.col("weight_phrases_raw").map_elements(
            normalize_weight_phrase_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("weight_ranges_norm")
    )
)

# Bước 2: chọn 1 weight_range duy nhất thành weight_group_final
# Ở đây, để an toàn, CHỈ dùng các dòng có đúng 1 khoảng cân nặng.
df_age_w = df_age_w.with_columns(
    pl.when(pl.col("weight_ranges_norm").list.len() == 1)
      .then(pl.col("weight_ranges_norm").list.first())
      .otherwise(None)
      .alias("weight_group_final")
)


In [41]:
df_age_w.head(1)

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new,gender_target_final,age_group_final,desc_all,weight_phrases_raw,weight_ranges_norm,weight_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str,str,str,list[str],list[str],str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cu…","""Không xác định""","""Từ 9M""","""không xác định chi tiết sản phẩm tên sản phẩm: hộp 2 núm ty drbrown's options pl…",[],[],null


In [42]:
df_labeled = df_age_w.filter(
    (pl.col("age_group_final") != "Không xác định") &
    (pl.col("weight_group_final").is_not_null())
)


In [43]:
weight_age_counts = (
    df_labeled
    .group_by(["weight_group_final", "age_group_final"])
    .agg(pl.count().alias("cnt"))
)

/tmp/ipykernel_4167176/456601868.py:4: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("cnt"))


In [44]:
total_by_weight = (
    weight_age_counts
    .group_by("weight_group_final")
    .agg(pl.col("cnt").sum().alias("total_cnt"))
)


In [45]:
weight_stats_full = (
    weight_age_counts
    .join(total_by_weight, on="weight_group_final", how="left")
    .with_columns(
        (pl.col("cnt") / pl.col("total_cnt")).alias("ratio")
    )
    .sort(["weight_group_final", "cnt"], descending=[False, True])
)


In [46]:
weight_stats_full


weight_group_final,age_group_final,cnt,total_cnt,ratio
str,str,u32,u32,f64
"""0-15KG""","""0-36M""",2,2,1.0
"""0-18KG""","""0-48M""",1,1,1.0
"""0-30KG""","""Từ 24M""",1,1,1.0
"""0-5KG""","""0-1M""",1,1,1.0
"""12-15KG""","""12-36M""",1,1,1.0
"""12-16KG""","""Từ 36M""",2,2,1.0
"""12-18KG""","""12-48M""",1,1,1.0
"""12-22KG""","""Từ 6M""",1,1,1.0
"""12KG+""","""12-48M""",1,1,1.0


Tiến hành fill

In [47]:
import polars as pl

# 1.1. Bắt cân nặng và chuẩn hóa cho toàn bộ df_age
df_age_w = (
    df_age
    .with_columns(
        pl.concat_str(
            [
                pl.col("description").fill_null(""),
                pl.col("description_new").fill_null(""),
            ],
            separator=" "
        )
        .str.to_lowercase()
        .alias("desc_all")
    )
    .with_columns(
        pl.col("desc_all").map_elements(
            extract_weight_phrases,
            return_dtype=pl.List(pl.Utf8)
        ).alias("weight_phrases_raw")
    )
    .with_columns(
        pl.col("weight_phrases_raw").map_elements(
            normalize_weight_phrase_list,
            return_dtype=pl.List(pl.Utf8)
        ).alias("weight_ranges_norm")
    )
)

# 1.2. Chọn 1 khoảng cân nặng duy nhất làm weight_group_final
# (Chỉ giữ những dòng có đúng 1 khoảng cân nặng, còn lại để null)
df_age_w = df_age_w.with_columns(
    pl.when(pl.col("weight_ranges_norm").list.len() == 1)
      .then(pl.col("weight_ranges_norm").list.first())
      .otherwise(None)
      .alias("weight_group_final")
)

In [48]:
# 2.1. Lấy các dòng có nhãn tuổi & có weight_group_final
df_labeled = df_age_w.filter(
    (pl.col("age_group_final") != "Không xác định") &
    (pl.col("weight_group_final").is_not_null())
)

# 2.2. Đếm số dòng theo cặp (weight_group_final, age_group_final)
weight_age_counts = (
    df_labeled
    .group_by(["weight_group_final", "age_group_final"])
    .agg(pl.count().alias("cnt"))
)

# 2.3. Tổng số dòng trong mỗi weight_group_final
total_by_weight = (
    weight_age_counts
    .group_by("weight_group_final")
    .agg(pl.col("cnt").sum().alias("total_cnt"))
)

# 2.4. Thêm ratio = cnt / total_cnt
weight_stats_full = (
    weight_age_counts
    .join(total_by_weight, on="weight_group_final", how="left")
    .with_columns(
        (pl.col("cnt") / pl.col("total_cnt")).alias("ratio")
    )
    .sort(["weight_group_final", "cnt"], descending=[False, True])
)

# Bạn có thể xem thử vài dòng:
print(weight_stats_full.head(5))


shape: (5, 5)
┌────────────────────┬─────────────────┬─────┬───────────┬───────┐
│ weight_group_final ┆ age_group_final ┆ cnt ┆ total_cnt ┆ ratio │
│ ---                ┆ ---             ┆ --- ┆ ---       ┆ ---   │
│ str                ┆ str             ┆ u32 ┆ u32       ┆ f64   │
╞════════════════════╪═════════════════╪═════╪═══════════╪═══════╡
│ 0-15KG             ┆ 0-36M           ┆ 2   ┆ 2         ┆ 1.0   │
│ 0-18KG             ┆ 0-48M           ┆ 1   ┆ 1         ┆ 1.0   │
│ 0-30KG             ┆ Từ 24M          ┆ 1   ┆ 1         ┆ 1.0   │
│ 0-5KG              ┆ 0-1M            ┆ 1   ┆ 1         ┆ 1.0   │
│ 12-15KG            ┆ 12-36M          ┆ 1   ┆ 1         ┆ 1.0   │
└────────────────────┴─────────────────┴─────┴───────────┴───────┘


/tmp/ipykernel_4167176/2629353381.py:11: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  .agg(pl.count().alias("cnt"))


In [49]:
# Đếm số nhãn khác nhau trong mỗi weight_group_final
weight_purity = (
    weight_stats_full
    .group_by("weight_group_final")
    .agg([
        pl.n_unique("age_group_final").alias("n_age_groups"),
        pl.col("total_cnt").first().alias("total_cnt"),
    ])
)

# Những weight_group_final chỉ có 1 nhãn tuổi -> pure (ratio = 1.0)
pure_weights = weight_purity.filter(pl.col("n_age_groups") == 1)

# Join lại để lấy age_group_final tương ứng
weight_mapping_strict_df = (
    pure_weights
    .join(weight_stats_full, on="weight_group_final", how="left")
    .select(["weight_group_final", "age_group_final", "total_cnt"])
    .unique()
)

# Dict: weight_group_final -> age_group_final
mapping_strict = {
    row["weight_group_final"]: row["age_group_final"]
    for row in weight_mapping_strict_df.iter_rows(named=True)
}

print("Số mapping strict:", len(mapping_strict))


Số mapping strict: 20


In [50]:
from collections import defaultdict

TOP_FRAC = 0.2   # giữ những nhãn có cnt >= 20% cnt lớn nhất
MAX_TOP  = 3     # tối đa 3 nhãn / 1 weight_group_final

# 3.2.1. Lấy các bucket có >1 nhãn (không thuần)
mixed_stats = (
    weight_stats_full
    .join(weight_purity, on="weight_group_final", how="left")
    .filter(pl.col("n_age_groups") > 1)
)

# 3.2.2. Thêm max_cnt cho mỗi weight_group_final
max_cnt_per_weight = (
    mixed_stats
    .group_by("weight_group_final")
    .agg(pl.col("cnt").max().alias("max_cnt"))
)

mixed_with_max = mixed_stats.join(max_cnt_per_weight, on="weight_group_final", how="left")

# 3.2.3. Làm top-K bằng Python (tránh dùng row_number/cumcount)
rows = list(mixed_with_max.iter_rows(named=True))

bucket_rows = defaultdict(list)
for row in rows:
    bucket_rows[row["weight_group_final"]].append(row)

mapping_topk = {}

for w, rlist in bucket_rows.items():
    # max_cnt theo weight_group_final
    max_cnt = max(r["cnt"] for r in rlist)
    threshold = TOP_FRAC * max_cnt

    # sort giảm dần theo cnt
    rlist_sorted = sorted(rlist, key=lambda r: -r["cnt"])

    # chọn các age_group có cnt >= threshold, giới hạn MAX_TOP
    selected = []
    for r in rlist_sorted:
        if r["cnt"] >= threshold:
            selected.append(r["age_group_final"])
        if len(selected) >= MAX_TOP:
            break

    mapping_topk[w] = selected

print("Ví dụ mapping_topk cho 4-8KG:", mapping_topk.get("4-8KG"))
print("Ví dụ mapping_topk cho 9-14KG:", mapping_topk.get("9-14KG"))


Ví dụ mapping_topk cho 4-8KG: ['3M-6M', 'Từ 4M', 'Từ 6M']
Ví dụ mapping_topk cho 9-14KG: ['12-36M', 'Từ 12M', '0-12M']


In [51]:
mapping_topk_str = {
    w: '["' + '", "'.join(ags) + '"]'
    for w, ags in mapping_topk.items()
}


In [52]:
df_age_fill_strict = (
    df_age_w
    .with_columns(
        pl.col("weight_group_final")
          .map_elements(lambda w: mapping_strict.get(w, None), return_dtype=pl.Utf8)
          .alias("age_from_weight_strict")
    )
    .with_columns(
        pl.when(
            (pl.col("age_group_final") == "Không xác định") &
            (pl.col("age_from_weight_strict").is_not_null())
        )
        .then(pl.col("age_from_weight_strict"))
        .otherwise(pl.col("age_group_final"))
        .alias("age_group_after_strict")
    )
    .with_columns(
        (pl.col("age_group_after_strict") != pl.col("age_group_final"))
        .alias("filled_from_weight_strict")
    )
)

print(
    "Số dòng được fill CHẮC CHẮN từ weight (ratio = 1.0):",
    df_age_fill_strict.filter(pl.col("filled_from_weight_strict")).height
)


Số dòng được fill CHẮC CHẮN từ weight (ratio = 1.0): 98


In [56]:
df_age_filled_all = (
    df_age_fill_strict
    .with_columns(
        pl.col("weight_group_final")
          .map_elements(lambda w: mapping_topk_str.get(w, None), return_dtype=pl.Utf8)
          .alias("age_from_weight_topk")
    )
    .with_columns(
        pl.when(
            (pl.col("age_group_after_strict") == "Không xác định") &
            (pl.col("age_from_weight_topk").is_not_null())
        )
        .then(pl.col("age_from_weight_topk"))
        .otherwise(pl.col("age_group_after_strict"))
        .alias("age_group_final_new")
    )
    .with_columns(
        (pl.col("age_group_final_new") != pl.col("age_group_final"))
        .alias("filled_from_weight_any")
    )
)

print(
    "Tổng số dòng được fill (strict + topK):",
    df_age_filled_all.filter(pl.col("filled_from_weight_any")).height
)

# Xem một vài dòng đã fill
df_age_filled_all.select([
    "item_id",
    "weight_group_final",
    "age_group_final",
    "age_group_after_strict",
    "age_from_weight_topk",
    "age_group_final_new",
    "filled_from_weight_strict",
    "filled_from_weight_any",
]).filter(pl.col("filled_from_weight_any")).head(20)


Tổng số dòng được fill (strict + topK): 614


item_id,weight_group_final,age_group_final,age_group_after_strict,age_from_weight_topk,age_group_final_new,filled_from_weight_strict,filled_from_weight_any
str,str,str,str,str,str,bool,bool
"""0020010000094""","""4-8KG""","""Không xác định""","""Không xác định""","""[""3M-6M"", ""Từ 4M"", ""Từ 6M""]""","""[""3M-6M"", ""Từ 4M"", ""Từ 6M""]""",false,true
"""0020010000098""","""6-11KG""","""Không xác định""","""12-36M""",null,"""12-36M""",true,true
"""0020010000151""","""0-5KG""","""Không xác định""","""0-1M""",null,"""0-1M""",true,true
"""0020010000396""","""15-28KG""","""Không xác định""","""Từ 6M""",null,"""Từ 6M""",true,true
"""0020010000096""","""9-14KG""","""Không xác định""","""Không xác định""","""[""12-36M"", ""Từ 12M"", ""0-12M""]""","""[""12-36M"", ""Từ 12M"", ""0-12M""]""",false,true
"""0020010000083""","""0-5KG""","""Không xác định""","""0-1M""",null,"""0-1M""",true,true
"""0014520040110""","""9-14KG""","""Không xác định""","""Không xác định""","""[""12-36M"", ""Từ 12M"", ""0-12M""]""","""[""12-36M"", ""Từ 12M"", ""0-12M""]""",false,true
"""0006040000099""","""9-14KG""","""Không xác định""","""Không xác định""","""[""12-36M"", ""Từ 12M"", ""0-12M""]""","""[""12-36M"", ""Từ 12M"", ""0-12M""]""",false,true
"""2294000000005""","""9-14KG""","""Không xác định""","""Không xác định""","""[""12-36M"", ""Từ 12M"", ""0-12M""]""","""[""12-36M"", ""Từ 12M"", ""0-12M""]""",false,true


In [58]:
# 1) Lấy mapping từ df_age_filled_all: item_id -> age_group_final_new
df_fill_map = (
    df_age_filled_all
    .select(["item_id", "age_group_final_new"])
    .unique(subset=["item_id"])  # phòng trường hợp trùng id (nếu có)
    .rename({"age_group_final_new": "age_group_from_weight"})
)

# 2) Join ngược vào df_age ban đầu theo item_id
df_age_updated = (
    df_age
    .join(df_fill_map, on="item_id", how="left")
    .with_columns(
        pl.when(
            (pl.col("age_group_final") == "Không xác định") &
            (pl.col("age_group_from_weight").is_not_null())
        )
        .then(pl.col("age_group_from_weight"))
        .otherwise(pl.col("age_group_final"))
        .alias("age_group_final_updated")
    )
)

# 3) (tuỳ chọn) xem thống kê số dòng được fill
n_filled = (
    df_age_updated
    .filter(pl.col("age_group_final_updated") != pl.col("age_group_final"))
    .height
)
print("Số dòng được fill thêm trong df_age:", n_filled)

# Ví dụ xem vài dòng
df_age_updated.select([
    "item_id",
    "age_group_final",
    "age_group_from_weight",
    "age_group_final_updated",
]).head(20)


Số dòng được fill thêm trong df_age: 614


item_id,age_group_final,age_group_from_weight,age_group_final_updated
str,str,str,str
"""0502020000004""","""Từ 9M""","""Từ 9M""","""Từ 9M"""
"""0010290040150""","""Từ 36M""","""Từ 36M""","""Từ 36M"""
"""0008010000015""","""0-12M""","""0-12M""","""0-12M"""
"""0020010000094""","""Không xác định""","""[""3M-6M"", ""Từ 4M"", ""Từ 6M""]""","""[""3M-6M"", ""Từ 4M"", ""Từ 6M""]"""
"""0020010000098""","""Không xác định""","""12-36M""","""12-36M"""
"""0024181040235""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0008180000017""","""0-12M""","""0-12M""","""0-12M"""
"""0020010000150""","""Không xác định""","""Không xác định""","""Không xác định"""
"""0501030000126""","""Không xác định""","""Không xác định""","""Không xác định"""


In [59]:
df_age_final = (
    df_age_updated
    .drop("age_group_final")
    .rename({"age_group_final_updated": "age_group_final"})
)

In [62]:
# df_age  = dataframe gốc trước khi fill
# df_age_final = dataframe sau khi fill đầy đủ (đã đổi sang age_group_final_updated)

def calc_unknown_ratio(df, colname="age_group_final"):
    total = df.height
    unknown = df.filter(pl.col(colname) == "Không xác định").height
    ratio = unknown / total * 100
    return total, unknown, ratio


# -------------------------------
# 1. Trước khi fill
# -------------------------------
total_before, unknown_before, ratio_before = calc_unknown_ratio(df_age, "age_group_final")
print("=== TRƯỚC KHI FILL ===")
print(f"Tổng số dòng: {total_before}")
print(f"Số dòng 'Không xác định': {unknown_before}")
print(f"Tỷ lệ: {ratio_before:.2f}%\n")


# -------------------------------
# 2. Sau khi fill
# -------------------------------
total_after, unknown_after, ratio_after = calc_unknown_ratio(df_age_final, "age_group_final")
print("=== SAU KHI FILL ===")
print(f"Tổng số dòng: {total_after}")
print(f"Số dòng 'Không xác định': {unknown_after}")
print(f"Tỷ lệ: {ratio_after:.2f}%\n")


# -------------------------------
# 3. Hiệu quả fill
# -------------------------------
print("=== HIỆU QUẢ FILL ===")
print(f"Giảm {unknown_before - unknown_after} dòng 'Không xác định'")
print(f"Giảm {ratio_before - ratio_after:.2f}% điểm phần trăm")

=== TRƯỚC KHI FILL ===
Tổng số dòng: 27323
Số dòng 'Không xác định': 11784
Tỷ lệ: 43.13%

=== SAU KHI FILL ===
Tổng số dòng: 27323
Số dòng 'Không xác định': 11170
Tỷ lệ: 40.88%

=== HIỆU QUẢ FILL ===
Giảm 614 dòng 'Không xác định'
Giảm 2.25% điểm phần trăm


In [65]:
df_age_final= df_age_final.drop("age_group_from_weight")
df_age_final.head(5)

item_id,price,category_l1,category_l2,category_l3,category,description,brand,gender_target,age_group,item_type,description_new,gender_target_final,age_group_final
str,"decimal[38,4]",str,str,str,str,str,str,str,str,str,str,str,str
"""0502020000004""",99000.0000,"""Babycare""","""Bình sữa, phụ kiện""","""Núm ty""","""Núm ty Dr Brown""","""Không xác định""","""Dr.Brown's""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Hộp 2 núm ty DrBrown's Options Plus cổ rộng Y cu…","""Không xác định""","""Từ 9M"""
"""0010290040150""",69000.0000,"""Thời trang""","""Cơ cấu hàng cũ""","""Thời trang bé trai, bé gái cũ""","""Bộ quần áo bé gái""","""Không xác định""","""Con Cưng""","""Bé Gái""","""Từ 3Y""","""Bộ quần áo""","""Không xác định""","""Bé Gái""","""Từ 36M"""
"""0008010000015""",45000.0000,"""Đồ chơi & Sách""","""0-1Y""","""Gặm nướu""","""Gặm nướu khác""","""- Chất liệu: Sản phẩm được làm bằng chất liệu silicone mềm, dẻo và nước đã được chưng cất đảm bảo an…","""Thương hiệu khác""","""Không xác định""","""Không xác định""","""Không xác định""","""Chi tiết sản phẩm Tên sản phẩm: Miếng Gặm Nướu Papa (CEQ004) (Cá hồng) C…","""Không xác định""","""0-12M"""
"""0020010000094""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Sơ Sinh""","""﻿﻿Tã dán Merries size S 82 miếng là sản phẩm dành cho bé 4-8kg đến từ thương hiệu uy tín Merries của…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""[""3M-6M"", ""Từ 4M"", ""Từ 6M""]"""
"""0020010000098""",401000.0000,"""Tã""","""Merries""","""Merries""","""Merries_Tã Quần""","""﻿﻿﻿Bỉm tã quần Merries size M 58 miếng là sản phẩm dành cho bé từ 6-11kg đến từ thương hiệu uy tín M…","""Merries Nhật""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""Không xác định""","""12-36M"""
